# Step 1 — 신문사 탐색 (Newspaper Discovery) v3
**변경점**: `/newspapers.json` 응답 구조를 직접 출력해서 진단 후, 올바른 키로 파싱.
여러 가능한 키(`newspapers`,`items`,`results`,`titles`,`data`)를 자동 시도.

## Cell 1 — 환경 설정

In [ ]:
!pip install -q ftfy
import requests, xml.etree.ElementTree as ET
import pandas as pd, time, re, json as _json
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import ftfy
from google.colab import drive
drive.mount('/content/drive')

def make_session():
    s=requests.Session()
    retry=Retry(total=2,connect=2,read=2,backoff_factor=0.5,
                status_forcelist=[429,500,502,503,504],
                allowed_methods=['GET'],raise_on_status=False)
    s.mount('https://',HTTPAdapter(max_retries=retry))
    s.mount('http://', HTTPAdapter(max_retries=retry))
    s.headers.update({'User-Agent':'KNU-causal-inference/1.0'})
    return s

SESSION=make_session()
REQUEST_TIMEOUT=(8,25)
RATE_LIMIT=0.25
CA_BASE='https://chroniclingamerica.loc.gov'
print('완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
Mounted at /content/drive
완료


## Cell 2 — 응답 구조 진단 + 주별 후보 추출
`/newspapers.json`의 실제 구조를 먼저 출력하고, 맞는 키/필드명으로 파싱.

In [ ]:
TARGET_STATES = {
    'west virginia': 'West Virginia',
    'illinois':      'Illinois',
    'new york':      'New York',
    'pennsylvania':  'Pennsylvania',
    'ohio':          'Ohio',
    'massachusetts': 'Massachusetts',
}

url = f'{CA_BASE}/newspapers.json'
r = SESSION.get(url, timeout=REQUEST_TIMEOUT)
print(f'요청: {url} -> HTTP {r.status_code}')
print(f'응답 바이트: {len(r.content)}')

data = r.json()
print(f'최상위 타입: {type(data)}')

if isinstance(data, dict):
    print(f'최상위 키: {list(data.keys())}')
    for k, v in data.items():
        if isinstance(v, list):
            print(f"  키 '{k}' -> 리스트, 길이 {len(v)}")
            if v:
                print(f"    첫 항목: {v[0]}")
        else:
            preview = str(v)[:200]
            print(f"  키 '{k}' -> {type(v).__name__}: {preview}")
    directory = None
    for key in ['newspapers','items','results','titles','data']:
        if isinstance(data.get(key), list) and data[key]:
            directory = data[key]
            print(f"\n→ '{key}' 키를 디렉토리로 사용 (길이 {len(directory)})")
            break
    if directory is None:
        directory = []
elif isinstance(data, list):
    print(f'최상위 리스트 길이: {len(data)}')
    if data:
        print(f'첫 항목: {data[0]}')
    directory = data
else:
    print(f'예상치 못한 타입. 원본 일부: {str(data)[:300]}')
    directory = []

print(f'\n전체 신문 디렉토리: {len(directory)}개')

# ── 후보 추출: state 필드명도 여러 가능성 시도 ──────────────────────────
candidates = []
if directory:
    sample = directory[0]
    print(f'\n샘플 항목의 키: {list(sample.keys()) if isinstance(sample,dict) else sample}')

    STATE_KEYS = ['state','State','place_of_publication','location']
    NAME_KEYS  = ['title','name','Title','Name']
    LCCN_KEYS  = ['lccn','LCCN']

    def get_first(d, keys):
        for k in keys:
            if k in d and d[k]:
                return d[k]
        return ''

    for item in directory:
        if not isinstance(item, dict): continue
        state_raw = str(get_first(item, STATE_KEYS)).strip().lower()
        lccn  = get_first(item, LCCN_KEYS)
        name  = get_first(item, NAME_KEYS)
        # state 필드가 'West Virginia' 외에 'va','wv' 등 다양할 수 있어 부분일치도 허용
        matched_state = None
        for k, v in TARGET_STATES.items():
            if k == state_raw or k in state_raw:
                matched_state = v; break
        if matched_state and lccn:
            candidates.append({'lccn': lccn, 'name': name, 'state': matched_state})

cand_df = pd.DataFrame(candidates, columns=['lccn','name','state']).drop_duplicates(subset='lccn')
print(f'\n총 후보 신문사: {len(cand_df)}개')
if len(cand_df) > 0:
    print(cand_df.groupby('state').size().to_string())
else:
    print('[경고] 후보 0개 — Cell 3은 기존 6개 신문사로만 진행됩니다.')

요청: https://chroniclingamerica.loc.gov/newspapers.json -> HTTP 200
응답 바이트: 60140
최상위 타입: <class 'dict'>
최상위 키: ['content.results']
  키 'content.results' -> 리스트, 길이 25
    첫 항목: {'calendar_url': {'class': 'calendar_url', 'icon': 'calendar_url', 'label': 'Browse Issue', 'screen_readers_only': True, 'url': 'https://www.loc.gov/item/sn85026945/?st=calendar', 'value': 'https://www.loc.gov/item/sn85026945/?st=calendar'}, 'campaigns': [], 'extract_timestamp': '2026-06-05T18:25:08.767Z', 'group': ['newspaper-titles', 'ndnp/newspapers', 'catalog-split-08', 'catalog', 'united-states-newspaper-directory', 'main-catalog-split-08'], 'id': 'http://www.loc.gov/item/sn85026945/', 'image_url': [], 'index': 1, 'language': {'class': 'language', 'label': 'English', 'value': 'english'}, 'latlong': [34.179102, -82.3777663], 'location_city': ['abbeville'], 'location_county': ['abbeville'], 'location_state': {'class': 'location_state', 'label': 'South Carolina', 'value': 'south carolina'}, 'location_str': 'Ab

## Cell 3 — 신문사당 5개 날짜 빠른 테스트

In [ ]:
MAX_PER_STATE = 8

KNOWN_GOOD = [
    {'lccn':'sn83030272','name':'The Sun','state':'New York'},
    {'lccn':'sn83045555','name':'Deseret Evening News','state':'Utah'},
    {'lccn':'sn84020645','name':'Montgomery Advertiser','state':'Alabama'},
    {'lccn':'sn85042462','name':'Los Angeles Herald','state':'California'},
    {'lccn':'sn85058130','name':'Salt Lake Herald','state':'Utah'},
    {'lccn':'sn83045604','name':'Washington Evening Star','state':'D.C.'},
]

if len(cand_df) > 0:
    limited = (cand_df.groupby('state', group_keys=False)
                       .apply(lambda g: g.head(MAX_PER_STATE)))
    new_candidates = limited.to_dict('records')
else:
    new_candidates = []

all_candidates = KNOWN_GOOD + new_candidates
seen_lccn = set()
dedup = []
for c in all_candidates:
    if c['lccn'] not in seen_lccn:
        seen_lccn.add(c['lccn'])
        dedup.append(c)
all_candidates = dedup
print(f'테스트할 신문사 총: {len(all_candidates)}개')

TEST_DATES = [
    ('1907-11-09', 'monongah_pre'),
    ('1907-12-07', 'monongah_post'),
    ('1909-11-14', 'cherry_post'),
    ('1911-03-03', 'triangle_pre'),
    ('1911-03-26', 'triangle_post'),
]

def extract_text(content_bytes):
    try:
        root = ET.fromstring(content_bytes)
    except: return ''
    ns = root.tag.split('}')[0].strip('{') if '}' in root.tag else ''
    P = f'{{{ns}}}' if ns else ''
    words=[s.get('CONTENT','') for s in root.iter(f'{P}String') if s.get('CONTENT','').strip()]
    text=' '.join(words)
    return ftfy.fix_text(text)

def ocr_quality(text):
    words=text.split()
    if not words: return 0
    lw=sum(1 for w in words if len(w)>=2)
    al=sum(1 for c in text if c.isalpha())
    return round((lw/len(words))*0.5+(al/max(len(text),1))*0.5,3)

def test_url(lccn, date_str, seq=1):
    url = f'{CA_BASE}/lccn/{lccn}/{date_str}/ed-1/seq-{seq}/ocr.xml'
    try:
        r = SESSION.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        if r.status_code==200 and b'<alto' in r.content[:1000].lower():
            text = extract_text(r.content)
            return True, len(text.split()), ocr_quality(text)
        return False, 0, 0
    except Exception:
        return False, 0, 0

print(f'\n신문사당 {len(TEST_DATES)}개 날짜 테스트 시작...\n')
results = []
for c in all_candidates:
    hits, wcs, ocrs = 0, [], []
    for date_str, label in TEST_DATES:
        ok, wc, ocr = test_url(c['lccn'], date_str)
        if ok:
            hits += 1
            wcs.append(wc); ocrs.append(ocr)
        time.sleep(RATE_LIMIT)
    avg_wc  = sum(wcs)/len(wcs) if wcs else 0
    avg_ocr = sum(ocrs)/len(ocrs) if ocrs else 0
    results.append({
        'lccn': c['lccn'], 'name': c['name'], 'state': c['state'],
        'hits': hits, 'total': len(TEST_DATES),
        'success_rate': hits/len(TEST_DATES),
        'avg_word_count': round(avg_wc), 'avg_ocr': round(avg_ocr,3),
    })
    status = '✅' if hits>=4 else ('△' if hits>=2 else '✗')
    print(f"  {status} {c['name'][:30]:30s} ({c['state'][:12]:12s}) "
          f"| {hits}/{len(TEST_DATES)} | OCR={avg_ocr:.2f}")

result_df = pd.DataFrame(results).sort_values('success_rate', ascending=False)

테스트할 신문사 총: 6개

신문사당 5개 날짜 테스트 시작...

  ✅ The Sun                        (New York    ) | 5/5 | OCR=0.88
  △ Deseret Evening News           (Utah        ) | 2/5 | OCR=0.84
  ✅ Montgomery Advertiser          (Alabama     ) | 5/5 | OCR=0.86
  △ Los Angeles Herald             (California  ) | 3/5 | OCR=0.87
  △ Salt Lake Herald               (Utah        ) | 2/5 | OCR=0.87
  ✗ Washington Evening Star        (D.C.        ) | 0/5 | OCR=0.00


## Cell 4 — 결과 정리 및 저장

In [ ]:
print('=== 신문사 탐색 결과 (성공률 순) ===\n')
print(result_df.to_string(index=False))

good = result_df[result_df['success_rate'] >= 0.8]
print(f"\n\n성공률 80% 이상: {len(good)}개")
print(good[['name','state','hits','avg_ocr']].to_string(index=False))

print('\n=== 주별 사용 가능 신문사 ===')
for state in ['West Virginia','Illinois','New York']:
    subset = good[good['state']==state]
    print(f"  {state}: {len(subset)}개")
    for _,r in subset.iterrows():
        print(f"    - {r['name']} ({r['lccn']})")

BASE = Path('/content/drive/MyDrive/경북대/인과추론')
out = BASE / '02_splits' / 'newspaper_discovery_v3.csv'
result_df.to_csv(out, index=False)
print(f'\n저장: {out}')

=== 신문사 탐색 결과 (성공률 순) ===

      lccn                    name      state  hits  total  success_rate  avg_word_count  avg_ocr
sn83030272                 The Sun   New York     5      5           1.0           11544    0.875
sn84020645   Montgomery Advertiser    Alabama     5      5           1.0            6148    0.863
sn85042462      Los Angeles Herald California     3      5           0.6            5025    0.873
sn83045555    Deseret Evening News       Utah     2      5           0.4            6945    0.843
sn85058130        Salt Lake Herald       Utah     2      5           0.4            8597    0.871
sn83045604 Washington Evening Star       D.C.     0      5           0.0               0    0.000


성공률 80% 이상: 2개
                 name    state  hits  avg_ocr
              The Sun New York     5    0.875
Montgomery Advertiser  Alabama     5    0.863

=== 주별 사용 가능 신문사 ===
  West Virginia: 0개
  Illinois: 0개
  New York: 1개
    - The Sun (sn83030272)

저장: /content/drive/MyDrive/경북대/인